# Common JSON Tools

Interactive JSON inspection tool for all files in `dcc/config/schemas`.


In [ ]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display


In [ ]:
try:
    # Check if running in an IPython environment (e.g., Colab, Jupyter)
    from IPython import get_ipython
    if 'IPKernelApp' in get_ipython().config:
        from IPython.display import display
        def print_or_display(obj):
            display(obj)
    else:
        def print_or_display(obj):
            print(obj)
except (ImportError, AttributeError):
    # Not in an IPython environment, use print
    def print_or_display(obj):
        print(obj)

print("Environment check complete. Using 'print_or_display' function.")

In [ ]:
# Locate schema JSON files under dcc/config/schemas (work from any CWD)
def find_schema_dir(start_path: Path):
    start = start_path.resolve()
    for base in [start, *start.parents]:
        candidates = [
            base / 'dcc' / 'config' / 'schemas',
            base / 'config' / 'schemas',
            base / 'schemas',
        ]
        for candidate in candidates:
            if candidate.exists() and candidate.is_dir():
                return candidate
    return None

schema_dir = find_schema_dir(Path.cwd())
if schema_dir is None:
    raise FileNotFoundError(f'No JSON schema files found in dcc/config/schemas from cwd {Path.cwd()} (searched parent paths)')

json_files = sorted(schema_dir.glob('*.json'))
if not json_files:
    raise FileNotFoundError(f'No JSON schema files found in {schema_dir}')

print('Resolved schema directory:', schema_dir)
print('Available JSON schema files:')
for i, fpath in enumerate(json_files, start=1):
    print(f'{i:>2}. {fpath.name}')

selection = input('Enter the index or file name to inspect (default 1): ').strip()
if selection == '':
    selection = '1'

if selection.isdigit():
    idx = int(selection) - 1
    if idx < 0 or idx >= len(json_files):
        raise IndexError('Selection index out of range')
    selected_path = json_files[idx]
else:
    selected_path = schema_dir / selection
    if not selected_path.exists():
        raise FileNotFoundError(f'File not found: {selected_path}')

print(f"\nLoading JSON file: {selected_path}")
with selected_path.open('r', encoding='utf-8') as f:
    json_data = json.load(f)

print('\nTop-level keys of loaded JSON:')
if isinstance(json_data, dict):
    print(list(json_data.keys()))
else:
    print(type(json_data))


In [ ]:
# This cell provides the requested table view.
# It summarizes the top-level keys and then provides a detailed, transposed
# table for any top-level key that contains a dictionary of objects.

def to_top_level_table(data):
    if not isinstance(data, dict):
        return pd.DataFrame([{'type': str(type(data)), 'value': data}])
    rows = []
    for key, value in data.items():
        value_type = type(value).__name__
        size = len(value) if hasattr(value, '__len__') else None
        preview = str(value)
        if len(preview) > 100:
            preview = preview[:100] + '...'
        rows.append({'key': key, 'type': value_type, 'size': size, 'preview': preview})
    return pd.DataFrame(rows)

def display_nested_properties(data):
    if not isinstance(data, dict):
        return
    print("\n--- Detailed Object Properties ---")
    found_table = False
    for key, value in data.items():
        if isinstance(value, dict) and value and all(isinstance(v, dict) for v in value.values()):
            found_table = True
            print(f"\nProperties for top-level key '{key}':")
            try:
                df = pd.DataFrame.from_dict(value, orient='index')
                with pd.option_context('display.max_rows', None, 'display.max_columns', None, 'display.width', 1000):
                    print_or_display(df)
            except Exception as e:
                print(f"Could not create a table for '{key}': {e}")
    if not found_table:
        print("No top-level keys containing a dictionary of objects were found to tabulate.")

# --- Main Execution ---
print('\nTop-Level Summary')
print_or_display(to_top_level_table(json_data))
display_nested_properties(json_data)
